# Lab 1, version 2 — Kamilya Islamova

## Result
The selected second solution achieves **macro-F1 0.6412**, QWK 0.5699,
and accuracy 0.6860 on the official 17,623-post test split.
The official baseline achieves 0.5410 macro-F1; my first submission achieves 0.6052.
The improvement over my first submission is 0.0360 absolute macro-F1.

## Connection to Lecture 2: Social Media User Profiling
- Slide 21: **SVD/LSA** compresses sparse TF-IDF into compact text features.
- Slides 32–33: **frozen sentence embeddings**, normalized to unit length,
  followed by a classifier. Here PCA reduces the 384-dimensional embeddings
  to 64 components before the joint tabular classifier.
- Slide 37: **early fusion** joins text and author features; **late fusion**
  combines predictions from models using different representations.
- Slides 22 and 49: earlier author activity provides context for the current post.

These lecture ideas motivate experiments; the slides do not guarantee a particular gain.

## Data and validation protocol
The released train/test split remains unchanged. Train contains 71,395 labelled
posts from 3,295 authors; test contains 17,623 posts from 824 different authors.
We compare candidates using three GroupShuffleSplit training/validation partitions,
with 20% of train authors reserved for validation, seeds 42, 123 and 2026.
Training and validation authors are disjoint within each split. These repeated
holdouts overlap with each other; they are not three independent test datasets.

All learned TF-IDF vocabularies, SVD/PCA transforms and classifiers fit only on the
fitting authors within each split. The pretrained encoder is frozen and performs
no fitting on this dataset. Test texts can be encoded by this fixed transform,
but no test labels are used for model or decision-rule selection.
The selected configuration maximizes mean validation macro-F1 across the three splits.
There are 60 candidate combinations, evaluated on each split.

The test set was already inspected during version 1. Version 2 selection uses only
training partitions, followed by one test evaluation of the frozen configuration.
This is an iterative benchmark evaluation, not a new untouched test set or a claim
of statistical significance.

## Additional author and attachment context
Version 1 features are retained. Nineteen new numerical features describe:
- current character/word counts relative to the previous five posts;
- dispersion, median, minimum and maximum of log reactions over recent history;
- the preceding post's reactions relative to the current historical level;
- preceding residual and mean of preceding residuals;
- how frequently this author previously used the current content type;
- image count, document page count, availability of document transcript,
  presence of a reshared post and a brand-partnership flag.

History excludes the current post. No equal timestamps within an author were
found; the reusable code rejects that case rather than assuming an arbitrary order.
Current-post reaction counts, comments, repost counts, residuals and labels are
not model inputs. They must not be used as shortcuts to the answer.
Historical reaction counts retain the baseline's crawl-time snapshot limitation.
The baseline's age-at-crawl and current follower-count features are also retained.

## Text representations and fusion
**Lexical model:** TF-IDF of word unigrams/bigrams (40,000 features, min_df=5),
then a balanced logistic regression. This is the version 1 text component.

**LSA early-fusion model:** fit 64-component TruncatedSVD to fitting-only TF-IDF,
normalize the resulting vectors, concatenate them with numerical/context features,
and fit the baseline's HistGradientBoostingClassifier.

**Semantic early-fusion model:** encode posts locally using
[sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2),
revision `e8f8c211226b894fcb81acc59f3b34ba3efd5f42`. Vectors have 384 dimensions
and unit norm. Fit 64-component PCA on the fitting authors only, concatenate with
context features, and fit HistGradientBoostingClassifier.
The encoder uses its 128-token limit: long posts are truncated. This is a limitation,
not an exhaustive test of neural text methods. Pseudonym placeholders are normalized.
No raw text is sent to an external inference service.

**Selected decision rule:** average the two early-fusion models equally, then
mix 85% of that average with 15% of the lexical classifier. Effective weights:
42.5% LSA/tabular, 42.5% semantic/tabular, 15% lexical. Multiply the typical-class
score by 1.6 and take the maximum. This adjusts decisions, not calibrated probabilities.

## Validation comparisons
Each candidate family below uses its best shared configuration by mean validation
macro-F1; it does not choose a separate configuration for each split.

| Family | Mean validation macro-F1 |
|---|---:|
| Submitted v1, fixed original configuration | 0.6185 |
| Added context + lexical late fusion | 0.6371 |
| LSA early fusion + lexical model | 0.6477 |
| Semantic early fusion + lexical model | 0.6441 |
| Context + separate semantic linear classifier | 0.6382 |
| Selected combination of LSA, semantic and lexical models | **0.6506** |

For the selected configuration, the three validation scores are
0.6501, 0.6536 and 0.6482. The corresponding v1 scores are 0.6176, 0.6209 and 0.6170.
Both the additional context and text representation contribute. The neural model
alone does not beat LSA in this experiment, but their combination improves the mean.
The new context features were tested as a group; we cannot attribute their entire
gain to any one feature.

## Official test results
| Model | Macro-F1 | QWK | Accuracy |
|---|---:|---:|---:|
| Official baseline | 0.5410 | 0.4576 | 0.5465 |
| Submitted v1 | 0.6052 | 0.5240 | 0.6444 |
| Version 2 | **0.6412** | **0.5699** | **0.6860** |

Per-class F1 (v1 → v2): below 0.4850 → 0.5185; typical 0.7085 → 0.7487;
above 0.6220 → 0.6565. All three improve, while below remains the weakest class.
The official evaluate.py verified all 17,623 predictions.

## Reproduction
The original requirements.txt suffices for training with cached embeddings.
Install requirements-v2.txt only when generating neural embeddings from scratch.
Use a Python environment with the project dependencies. Tested ML environment:
Python 3.14.8, scikit-learn 1.9.1. The optional encoder was run with
sentence-transformers 6.2.0, transformers 5.20.2 and torch 2.14.1 on Apple MPS.

From the repository root:
```bash
python -m pip install -r requirements-v2.txt
python improved_v2.py --stage prepare
python encode_posts_v2.py
python improved_v2.py --stage validate --embeddings results/v2_cache/embeddings.npy
python improved_v2.py --stage final --embeddings results/v2_cache/embeddings.npy
python evaluate.py --predictions lab1_submission_v2/predictions.csv
```
Model caches, intermediate tables and text stay under results/, which the original
.gitignore excludes. Submit code, this notebook and lab1_submission_v2/ only.
The cells below display the saved actual results; they do not retrain the models.


In [1]:
import json
from pathlib import Path
results = json.loads(Path("lab1_submission_v2/metrics.json").read_text())
print(json.dumps(results["test"], indent=2))


{
  "macro_f1": 0.6412,
  "qwk": 0.5699,
  "accuracy": 0.686,
  "f1_by_class": [
    0.5184969414506263,
    0.7487003341997772,
    0.6565242832065535
  ],
  "confusion_matrix": [
    [
      1780,
      1719,
      42
    ],
    [
      1467,
      8065,
      959
    ],
    [
      78,
      1269,
      2244
    ]
  ]
}
